This notebook will be used to reimplement the CoastSeg Segformer model in PyTorch, where the b0 backbone was utilized with Dice loss

In [ ]:
# TODO: Look into adding Log-Cosh Dice and Haussdorf (or another boundary aware loss function?)
# TODO: try adding the sobel edge loss to other loss functions for improved edge/boundary preservation?

# TODO: look into uploading trained models to HF. Reference: https://huggingface.co/learn/cookbook/en/semantic_segmentation_fine_tuning_inference 

%reload_ext autoreload
%autoreload 2

import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # SET_GPU
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0" 
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
import json

from pathlib import Path
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from transformers import SegformerForSemanticSegmentation   # TODO: Try out Mask2Former in a future project. It out performed segformer in https://doi.org/10.3390/rs18020325
from datetime import datetime

if torch.cuda.is_available():
    device = "cuda" 
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = "mps"
else:
    device = "cpu"

sys.path.insert(0, os.path.abspath('../../src'))
from src import dataset_loader, training_help

from src.loss_functions import (
    FocalLoss,
    DiceLoss,
    GeneralizedDiceLoss,
    TverskyLoss,
    FocalTverskyLoss,
    AsymmetricUnifiedFocalLoss,
    ComboLoss,
    LovaszSoftmaxLoss
)

print(f"Using device: {device}")

In [ ]:
with open(Path.cwd() / 'ini_training_config.json') as config_data:
    config_dict = json.load(config_data)

class Config:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

config = Config(**config_dict)

In [ ]:
# TODO: Make these command line arguments when running the train.py script from the terminal

config.SEED = 42                            # NOTE: Testing five seeds 42, 7, 300, 1337, 12345       
config.MODEL_NAME = "nvidia/mit-b0"         # NOTE: this goes 0-5, and gets increasing larger
config.LOSS = "wCCE"               # NOTE: Options are CCE, wCCE, Focal, Dice, GenDice, Combo, Tversky, FocalTversky, AsymmetricUnifiedFocal, LovaszSoftmax
config.IMAGES_DIR = Path("/Users/clayc/Documents/Work/HazShoalWaves/TrainDatasets/images_and_labels_RGB/images")     # mac
config.LABELS_DIR = Path("/Users/clayc/Documents/Work/HazShoalWaves/TrainDatasets/images_and_labels_RGB/labels")     # mac

config.OUTPUT_DIR = Path(f"../../trained_models/{config.MODEL_NAME}_{config.SEED}_{config.LOSS}_{datetime.now().strftime("%Y_%m_%d_%H_%M_%S")}")       # TODO: CHANGE THE OUTPUT DIRECTORY

In [ ]:
LOSSES = {
    "CCE":     nn.CrossEntropyLoss(ignore_index=255),
    "wCCE":     nn.CrossEntropyLoss(weight = None, ignore_index=255),                   # NOTE: weight tensor determined using median freq
    "Focal":  FocalLoss(gamma=2.0, weight=None, ignore_index=255, reduction='mean'),
    "Dice":   DiceLoss(config.NUM_CLASSES, smooth=1.),
    'GenDice': GeneralizedDiceLoss(config.NUM_CLASSES, weight_mode='batch', smooth=1.),
    "Combo": ComboLoss(config.NUM_CLASSES, beta=0.6, alpha=0.5, smooth=1.),
    "Tversky":TverskyLoss(alpha=0.3, beta=0.7, smooth=1.),
    "FocalTversky": FocalTverskyLoss(alpha=0.3, beta=0.7, gamma=0.75, smooth=1.),
    "AsymmetricUnifiedFocal": AsymmetricUnifiedFocalLoss(config.NUM_CLASSES, rare_classes=[1,2], lam=0.5, delta=0.6, gamma=0.5, smooth=1.),
    "LovaszSoftmax": LovaszSoftmaxLoss(config.NUM_CLASSES, per_image=True)
}

criterion = LOSSES[config.LOSS]
np.random.seed(config.SEED)

torch.manual_seed(config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.SEED)

## Load Data and Create DataLoaders

In [ ]:
if (Path.cwd() / 'data' / 'train_loader.pt').exists():
    train_loader = torch.load(Path.cwd() / 'data' / 'train_loader.pt', weights_only=False)
    test_loader = torch.load(Path.cwd() / 'data' / 'test_loader.pt', weights_only=False)
    val_loader = torch.load(Path.cwd() / 'data' / 'val_loader.pt', weights_only=False)

else:
    image_paths = sorted([
        str(config.IMAGES_DIR / f) 
        for f in os.listdir(config.IMAGES_DIR) 
        if f.endswith(('.jpg', '.jpeg', '.png'))
    ])
    
    label_paths = [
        Path(config.LABELS_DIR / f"{Path(im).stem}_label.jpg") 
        for im in image_paths
    ]
    
    # Verify all label files exist
    valid_pairs = [(img, lbl) for img, lbl in zip(image_paths, label_paths) if os.path.exists(lbl)]
    image_paths, label_paths = zip(*valid_pairs) if valid_pairs else ([], [])
        
    # Train/validation split
    X_train, X_test, y_train, y_test = train_test_split(
        list(image_paths),
        list(label_paths),
        test_size=1-config.VAL_SPLIT,
        random_state=config.SEED,
        shuffle=True
    )
    
    X_test, X_val, y_test, y_val = train_test_split(
        X_test,
        y_test,
        test_size=0.5,
        random_state=config.SEED,
        shuffle=True
    )
    
    processor = dataset_loader.build_processor()
    
    train_ds = dataset_loader.CoastalSegDataset(X_train, y_train,
                                 num_classes=config.NUM_CLASSES, tile_size=config.TILE_SIZE,
                                 processor=processor, augment=True)
    
    test_ds = dataset_loader.CoastalSegDataset(X_test, y_test,
                                 num_classes=config.NUM_CLASSES, tile_size=config.TILE_SIZE,
                                 processor=processor, augment=False)

    val_ds = dataset_loader.CoastalSegDataset(X_val, y_val,
                                 num_classes=config.NUM_CLASSES, tile_size=config.TILE_SIZE,
                                 processor=processor, augment=False)

    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True,
                        num_workers=os.cpu_count() - 4, pin_memory=True)

    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=os.cpu_count() - 4, pin_memory=True)

    val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                        num_workers=os.cpu_count() - 4, pin_memory=True)

    torch.save(train_loader, Path.cwd() / 'data' / 'train_loader.pt')
    torch.save(test_loader, Path.cwd() / 'data' / 'test_loader.pt')
    torch.save(val_loader, Path.cwd() / 'data' / 'val_loader.pt')

    del X_train, y_train, X_test, X_val, y_test, y_val, train_ds, test_ds, val_ds

In [ ]:
# # TODO: Make sure some of Jonathan's labeled FRF imagery is in the test set. We can use them for some examples, and the shoreline accuracy example if we just plug in new models directly into CoastSeg

# Extra labels from Jonathan Markel, 23072026
# NOTE: I only had these saved on my windows PC, and now have no way to access them. So going to stick with Buscombe's dataset(s)
# frf_images = Path("/mnt/c/Users/clayc/Documents/WhiteWaterClassifier/coastseg_frf_labels/dataset/images")
# frf_labels = Path("/mnt/c/Users/clayc/Documents/WhiteWaterClassifier/coastseg_frf_labels/dataset/labels")

# frf_label_jpegs = sorted(dataset_loader.batch_convert_labels(frf_labels, frf_labels / 'pngs'))
# frf_images_pngs = [frf_images / (im.name[:-4] + '.png') for im in frf_label_jpegs]

# # Get image and label paths
# usgs_image_paths = sorted([
#     str(config.IMAGES_DIR / f) 
#     for f in os.listdir(config.IMAGES_DIR) 
#     if f.endswith(('.jpg', '.jpeg', '.png'))
# ])

# image_paths = usgs_image_paths + frf_images_pngs

# usgs_label_paths = [
#     Path(config.LABELS_DIR / f"{Path(im).stem}_label.jpg") 
#     for im in usgs_image_paths
# ]

# label_paths = usgs_label_paths + frf_label_jpegs

# # Verify all label files exist
# valid_pairs = [(img, lbl) for img, lbl in zip(image_paths, label_paths) if os.path.exists(lbl)]
# image_paths, label_paths = zip(*valid_pairs) if valid_pairs else ([], [])

# print(f"Found {len(image_paths)} valid image-label pairs")

# # Train/validation split
# X_train, X_test, y_train, y_test = train_test_split(
#     list(image_paths),
#     list(label_paths),
#     test_size=1-config.VAL_SPLIT,
#     random_state=config.SEED,
#     shuffle=True
# )

# X_test, X_val, y_test, y_val = train_test_split(
#     X_test,
#     y_test,
#     test_size=0.5,
#     random_state=config.SEED,
#     shuffle=True
# )

# print(f"Training samples: {len(X_train)}")
# print(f"Testing samples: {len(X_test)}")
# print(f"Validation samples: {len(X_val)}")

# processor = dataset_loader.build_processor()

# train_ds = dataset_loader.CoastalSegDataset(X_train, y_train,
#                              num_classes=config.NUM_CLASSES, tile_size=config.TILE_SIZE,
#                              processor=processor, augment=True)

# test_ds = dataset_loader.CoastalSegDataset(X_test, y_test,
#                              num_classes=config.NUM_CLASSES, tile_size=config.TILE_SIZE,
#                              processor=processor, augment=False)

# val_ds = dataset_loader.CoastalSegDataset(X_val, y_val,
#                              num_classes=config.NUM_CLASSES, tile_size=config.TILE_SIZE,
#                              processor=processor, augment=False)

# train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True,
#                     num_workers=4, pin_memory=True)

# test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, shuffle=False,
#                     num_workers=4, pin_memory=True)

# val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False,
#                     num_workers=4, pin_memory=True)

# torch.save(train_ds, Path.cwd().parent.parent / 'data' / 'train_dataset.pth')
# torch.save(test_ds, Path.cwd().parent.parent / 'data' / 'test_dataset.pth')
# torch.save(val_ds, Path.cwd().parent.parent / 'data' / 'val_dataset.pth')

# torch.save(train_loader, Path.cwd().parent.parent / 'data' / 'train_loader.pth')
# torch.save(test_loader, Path.cwd().parent.parent / 'data' / 'test_loader.pth')
# torch.save(val_loader, Path.cwd().parent.parent / 'data' / 'val_loader.pth')

# batch = next(iter(train_loader))
# print("pixel_values:", batch["pixel_values"].shape)  # (B, 3, 512, 512)
# print("labels:      ", batch["labels"].shape)        # (B, 512, 512)
# print("label range: ", batch["labels"].min().item(),
#       "->", batch["labels"].max().item())


In [ ]:
# weights should be -> [0.2854, 8.7893, 5.4946, 0.5501]

if config.LOSS == 'wCCE':
    class_counts, suggested_weights = dataset_loader.analyze_class_distribution(config, train_loader.dataset.label_paths)

    class_weights = torch.as_tensor(suggested_weights, dtype=torch.float32, device=device)
    config.LOSS_WEIGHTS = class_weights


    # replace the criterion which was established in the second code cell, since that was initialized without the determined class weights
    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)
    display(criterion.weight)
else:
    print(f"Loss function is {config.LOSS}, so heursitic class weighting will not be applied")

## Visualize Sample Data
- New Samples from Oregon, Folly Beach, Barnegat last

In [ ]:
import random

# Generate 3 random indices
random_indices = random.sample(range(len(train_loader.dataset)), min(3, len(train_loader.dataset)))

for i in random_indices:
    dataset_loader.visualize_sample(config, train_loader.dataset, i, show_augmented=True)

# Establish Learning Rate Scheduler and Optimizer

In [ ]:
model = SegformerForSemanticSegmentation.from_pretrained(
    config.MODEL_NAME,
    num_labels=config.NUM_CLASSES,
    id2label={i: str(i) for i in range(config.NUM_CLASSES)},
    label2id={str(i): i for i in range(config.NUM_CLASSES)},
    classifier_dropout_prob=config.DROPOUT,
    hidden_dropout_prob=config.DROPOUT,
).to(device)

In [ ]:
def lr_lambda(epoch):
    if epoch < config.RAMPUP_EPOCHS:
        lr = (config.MAX_LR - config.START_LR) / config.RAMPUP_EPOCHS * epoch + config.START_LR
    elif epoch < config.RAMPUP_EPOCHS + config.SUSTAIN_EPOCHS:
        lr = config.MAX_LR  
    else:
        lr = (config.MAX_LR - config.MIN_LR) * config.EXP_DECAY ** (
              epoch - config.RAMPUP_EPOCHS - config.SUSTAIN_EPOCHS) + config.MIN_LR
    return lr / config.MAX_LR   # LambdaLR multiplies base_lr

optimizer = torch.optim.AdamW(model.parameters(), lr=config.MAX_LR)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Training Loop

In [ ]:
# NOTE: Just an observation, b0 backbone is typically maxing out around 0.75 mIoU on validation dataset
# Once I get access to a bigger computer, I train larger backbone. Maybe make better?

results, history, test_metrics = training_help.train(model, train_loader, val_loader, test_loader, criterion, optimizer, scheduler, config, device=device)
print(results)

# Training plot history

In [ ]:
training_help.training_history_plot(config, history)

In [ ]:
# just some extra plots for each of the 4 classes if we want to try them out
# would use the index for the best model

# training_help.plot_per_class_metric(config, history, 'Precision', best_idx)
# training_help.plot_per_class_metric(config, history, 'Recall', best_idx)
# training_help.plot_per_class_metric(config, history, 'IoU', best_idx)
# training_help.plot_per_class_metric(config, history, 'F1-Score', best_idx)

# Export Files for Inference
- May be worth looking into uploading to Hugging Face: https://huggingface.co/learn/cookbook/en/semantic_segmentation_fine_tuning_inference

In [ ]:
config_dict = {k: v for k, v in vars(config.__class__).items() if k.isupper()}

with open(config.OUTPUT_DIR / 'model_config.json', 'w') as config_file:
        json.dump(config_dict, config_file, indent=4, default=str)

with open(config.OUTPUT_DIR / 'model_history.json', 'w') as history_file:
        json.dump(history, history_file, indent=4, default = float)

with open(config.OUTPUT_DIR / 'test_metrics.json', 'w') as test_metrics_file:
        json.dump(test_metrics, test_metrics_file, indent=4, default = float)